# 07 · Model dispatch, physical returns and stock legs

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("fulfillment", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


A dispatch header has item rows. One order line can split across deliveries. The ERPNext line points to its native Sales Order Item. The dealer fallback requires exactly one matching line by order and product. A warehouse on the order is planned. The warehouse on the dispatch is actual.

Movement grain is one signed leg per product and location. This slice models the verified ERPNext Stock Entry legs. Review warehouse and quantity coverage before treating this as a stock balance dataset.

In [2]:
display(frame(model, "shipment"), frame(model, "shipment_line"))
display(frame(model, "return_document"), frame(model, "return_line"))
display(frame(model, "inventory_movement_line"))
print(model.coverage["inventory_movement_line"])
display(pd.DataFrame(model.exceptions))

,id,legal_entity_id,source_id,sales_order_id,posting_date,docstatus,original_document_source_id
0,"[""parent"",""MAT-DN-2026-00001""]",parent,MAT-DN-2026-00001,"[""parent"",""SAL-ORD-2026-00001""]",2026-11-03,1.0,None
1,"[""parent"",""MAT-DN-2026-00002""]",parent,MAT-DN-2026-00002,"[""parent"",""SAL-ORD-2026-00001""]",2026-11-05,1.0,None
2,"[""parent"",""MAT-DN-2026-00003""]",parent,MAT-DN-2026-00003,"[""parent"",""SAL-ORD-2026-00006""]",2026-11-15,1.0,None
3,"[""dealer"",""D-BRONZE-R2-SHIP""]",dealer,D-BRONZE-R2-SHIP,"[""dealer"",""D-BRONZE-R2-ORDER""]",None,NaN,None
4,"[""dealer"",""D-DEL1""]",dealer,D-DEL1,"[""dealer"",""D-S1""]",None,NaN,None


,id,legal_entity_id,document_id,source_id,source_product_id,warehouse_id,shipment_line_id,sales_order_line_id,quantity,uom
0,"[""parent"",""MAT-DN-2026-00001"",""vqnei0r1v0""]",parent,"[""parent"",""MAT-DN-2026-00001""]",vqnei0r1v0,"[""parent"",""P-PUMP""]","[""parent"",""P-WH1 - OPD""]",None,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]",6.0,Nos
1,"[""parent"",""MAT-DN-2026-00002"",""vqtf30brbp""]",parent,"[""parent"",""MAT-DN-2026-00002""]",vqtf30brbp,"[""parent"",""P-PUMP""]","[""parent"",""P-WH2 - OPD""]",None,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]",4.0,Nos
2,"[""parent"",""MAT-DN-2026-00003"",""brlofkakoo""]",parent,"[""parent"",""MAT-DN-2026-00003""]",brlofkakoo,"[""parent"",""P-PUMP""]","[""parent"",""P-WH1 - OPD""]",None,"[""parent"",""SAL-ORD-2026-00006"",""brbeshsujo""]",2.0,Nos
3,"[""dealer"",""D-BRONZE-R2-SHIP"",""D-BRONZE-R2-SHIP""]",dealer,"[""dealer"",""D-BRONZE-R2-SHIP""]",D-BRONZE-R2-SHIP,"[""dealer"",""D-PUMP""]","[""dealer"",""D-WH1""]",None,"[""dealer"",""D-BRONZE-R2-ORDER"",""D-BRONZE-R2-LINE""]",2,Each
4,"[""dealer"",""D-DEL1"",""D-DEL1""]",dealer,"[""dealer"",""D-DEL1""]",D-DEL1,"[""dealer"",""D-PUMP""]","[""dealer"",""D-WH1""]",None,"[""dealer"",""D-S1"",""D-L1""]",1,Each


""


""


,id,legal_entity_id,source_document_id,source_product_id,warehouse_id,quantity,uom,posting_date,docstatus
0,"[""parent"",""Stock Entry"",""MAT-STE-2026-00001"",""...",parent,MAT-STE-2026-00001,"[""parent"",""P-PUMP""]","[""parent"",""P-WH1 - OPD""]",6.0,Nos,2026-11-01,1
1,"[""parent"",""Stock Entry"",""MAT-STE-2026-00002"",""...",parent,MAT-STE-2026-00002,"[""parent"",""P-PUMP""]","[""parent"",""P-WH2 - OPD""]",4.0,Nos,2026-11-01,1
2,"[""parent"",""Stock Entry"",""MAT-STE-2026-00003"",""...",parent,MAT-STE-2026-00003,"[""parent"",""P-PUMP""]","[""parent"",""P-WH1 - OPD""]",2.0,Nos,2026-11-15,1


ERPNext Stock Entry legs only. Shipment and return movements require verified stock ledger coverage.


""
